# Notebook Colab per Training Diffusion
Assicurati di impostare l'acceleratore hardware su GPU (T4).

In [ ]:
# Controlliamo la disponibilità della GPU
!nvidia-smi

# Montiamo Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os

# Cambiamo directory di lavoro per far sì che gli import funzionino
%cd /content/vessel-project
sys.path.append('/content/vessel-project')

In [ ]:
import torch
from torch.utils.data import DataLoader

# Importiamo i file che abbiamo appena creato
from src.dataset import VesselDataset # Sostituisci col nome reale della tua classe dataset se diverso
from src.models.diffusion_scheduler import DiffusionScheduler
from src.models.diffusion_unet import ConditionalUNet
from src.engine_diffusion import train_diffusion

In [ ]:
# Configurazione Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso: {DEVICE}")

# 1. Dataset & DataLoader
train_dataset = VesselDataset(
    images_dir='path/to/train/images', 
    masks_dir='path/to/train/masks', 
    patch_size=128
)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)

# 2. Inizializziamo lo Scheduler e la U-Net Leggera
scheduler = DiffusionScheduler(num_timesteps=1000, device=DEVICE)
model = ConditionalUNet(image_channels=3, mask_channels=1, base_dim=32).to(DEVICE)

# 3. Ottimizzatore
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [ ]:
# Scegliamo una cartella persistente su Google Drive dove salvare il modello
DRIVE_SAVE_DIR = '/content/drive/MyDrive/Vessel_Diffusion_Checkpoints'

# Lanciamo l'addestramento
train_diffusion(
    model=model,
    dataloader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=50,
    device=DEVICE,
    save_dir="./local_checkpoints",
    drive_save_dir=DRIVE_SAVE_DIR
)

In [ ]:
import matplotlib.pyplot as plt
from src.models.diffusion_inference import DiffusionPipeline

# Carica i pesi migliori appena salvati
model.load_state_dict(torch.load(f"{DRIVE_SAVE_DIR}/diffusion_best.pth"))
pipeline = DiffusionPipeline(model=model, scheduler=scheduler, device=DEVICE)

# Test Inferenza su Immagine Fittizia
test_image_tensor = torch.rand((1, 3, 1024, 1024)).to(DEVICE)
final_mask = pipeline.infer_full_image(test_image_tensor, patch_size=128, stride=64)

# Plot dei risultati
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(test_image_tensor[0].cpu().permute(1, 2, 0).numpy())
axes[0].set_title("Fondo Oculare Originale")
pred_binary = (final_mask[0, 0].cpu().numpy() > 0.5).astype('uint8')
axes[1].imshow(pred_binary, cmap='gray')
axes[1].set_title("Maschera Predetta")
plt.show()